In [ ]:
!uv pip install --no-cache-dir --force-reinstall "tinker-cookbook @ git+https://github.com/thinking-machines-lab/tinker-cookbook.git@nightly"

In [ ]:
# === DIAGNOSTIC: how much does the rank-64 -> rank-32 SVD truncation actually discard? ===
# Reads the RAW Tinker adapter (the input to build_lora_adapter, NOT the converted output).
# For each Mamba layer it reconstructs the official block-diagonal merged delta
#   D = blockdiag(B_gate @ A_gate, B_x @ A_x)   (true rank up to 64, shared input columns)
# and reports the Frobenius ENERGY retained by the best rank-32 approximation:
#   keepF% = sum(sigma^2[:32]) / sum(sigma^2)      <- the load-bearing number
# Singular values are computed exactly via QR-then-SVD on the 64x64 core (never
# materializes the 10304x2688 delta). Decision: >=97% ignore | 90-97% A/B test | <90% fix.
import re, glob, math, torch
from safetensors import safe_open

ADAPTER_DIR = "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20"
TARGET_RANK = 32  # competition hard cap (vLLM max_lora_rank=32)

st_files = sorted(glob.glob(f"{ADAPTER_DIR}/**/adapter_model.safetensors", recursive=True))
assert st_files, f"No adapter_model.safetensors under {ADAPTER_DIR}"
ST = st_files[0]
print(f"adapter: {ST}\n")

T = {}
with safe_open(ST, framework="pt", device="cpu") as f:
    for k in f.keys():
        T[k] = f.get_tensor(k)

# discover gate_proj / x_proj lora pairs by pattern (robust to the exact key prefix)
bases = sorted({re.sub(r"\.lora_[AB]\.weight$", "", k) for k in T if ".lora_" in k})
layers = {}
for b in bases:
    for proj in ("gate_proj", "x_proj"):
        if b.endswith(f".{proj}"):
            layers.setdefault(b[: -len(f".{proj}")], {})[proj] = b
layers = {lp: d for lp, d in layers.items() if "gate_proj" in d and "x_proj" in d}
if not layers:
    leafs = sorted({b.rsplit(".", 1)[-1] for b in bases})
    raise AssertionError(
        "No gate_proj/x_proj pairs found. Sample keys:\n  "
        + "\n  ".join(list(T)[:8])
        + f"\nLeaf module names present: {leafs}"
    )
print(f"found {len(layers)} fused Mamba in_proj layers\n")


def block_spectrum(B, A):
    # exact singular values of (B @ A) via QR: B=Qb Rb, A^T=Qa Ra -> svd(Rb @ Ra^T)
    B, A = B.float(), A.float()
    _, Rb = torch.linalg.qr(B)
    _, Ra = torch.linalg.qr(A.T)
    return torch.linalg.svdvals(Rb @ Ra.T)


def principal_angles(Ag, Ax):
    # angles (deg) between rowspan(Ag) and rowspan(Ax); small => high input-subspace overlap
    Qg = torch.linalg.qr(Ag.float().T).Q
    Qx = torch.linalg.qr(Ax.float().T).Q
    cos = torch.linalg.svdvals(Qg.T @ Qx).clamp(-1, 1)
    return torch.rad2deg(torch.arccos(cos))


rows, agg_kept, agg_tot = [], 0.0, 0.0
for lp in sorted(layers):
    gp, xp = layers[lp]["gate_proj"], layers[lp]["x_proj"]
    Ag, Bg = T[f"{gp}.lora_A.weight"].float(), T[f"{gp}.lora_B.weight"].float()
    Ax, Bx = T[f"{xp}.lora_A.weight"].float(), T[f"{xp}.lora_B.weight"].float()
    r, out_g, out_x = Ag.shape[0], Bg.shape[0], Bx.shape[0]

    A_cat = torch.cat([Ag, Ax], dim=0)               # (2r, hidden)
    B_blk = torch.zeros(out_g + out_x, 2 * r)
    B_blk[:out_g, :r] = Bg
    B_blk[out_g:, r:] = Bx                            # block-diagonal
    S = block_spectrum(B_blk, A_cat)                  # exact singular values of merged delta

    s2 = (S.double() ** 2)
    tot = s2.sum().item()
    keepF = s2[:TARGET_RANK].sum().item() / tot if tot > 0 else 1.0
    rel_err = math.sqrt(max(0.0, 1.0 - keepF))
    eff_rank = (S.sum() ** 2 / (S ** 2).sum()).item()
    mass_g = (block_spectrum(Bg, Ag) ** 2).sum().item()
    mass_x = (block_spectrum(Bx, Ax) ** 2).sum().item()
    share_g = 100 * mass_g / (mass_g + mass_x)
    ang = principal_angles(Ag, Ax)

    agg_kept += keepF * tot
    agg_tot += tot
    rows.append((lp.split("layers.")[-1][:18], 100 * keepF, 100 * rel_err,
                 eff_rank, share_g, ang.mean().item(), ang.min().item()))

print(f"{'layer':>18} {'keepF%':>7} {'relErr%':>8} {'effRank':>8} {'gateM%':>7} {'ang_mu':>7} {'ang_min':>8}")
for x in rows:
    print(f"{x[0]:>18} {x[1]:7.2f} {x[2]:8.2f} {x[3]:8.2f} {x[4]:7.1f} {x[5]:7.1f} {x[6]:8.1f}")

agg = 100 * agg_kept / agg_tot
keeps = [x[1] for x in rows]
print("\n=== AGGREGATE ===")
print(f"layers measured            : {len(rows)}")
print(f"mass-weighted retained F%  : {agg:6.2f}%   (energy kept at rank {TARGET_RANK})")
print(f"per-layer keepF%           : min {min(keeps):.2f}  mean {sum(keeps)/len(keeps):.2f}  max {max(keeps):.2f}")
print(f"aggregate rel recon error  : {math.sqrt(max(0.0, 1 - agg/100))*100:6.2f}%")
print(f"\nDECISION: >=97% ignore | 90-97% A/B test rank-64 model vs shipped | <90% reclaim (activation-aware SVD or native rank-32)")


In [ ]:
# === CALIBRATION (GPU): per-layer in_proj input covariance C = E[x x^T] for activation-aware SVD ===
# Loads the base model, hooks each Mamba mixer.in_proj input, feeds a stratified sample of the REAL
# training traces (already tokenized in snapshot 04-08-16-14), accumulates x x^T per layer, and saves
# to COV_OUT. The converter cell below whitens its rank-32 truncation by these covariances => it
# minimizes FUNCTIONAL output error E||(D-D_hat) x||^2 instead of raw weight error ||D-D_hat||_F.
# SKIP this cell to fall back to plain (weight-Frobenius) SVD -- the converter auto-detects COV_OUT.
import os, json, glob, random, collections, re, torch
from transformers import AutoModelForCausalLM

BASE_MODEL   = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"  # offline Kaggle: set to the attached model path
COV_OUT      = "/kaggle/working/in_proj_cov.pt"
PER_CATEGORY = 40     # stratified sequences per category (9 cats -> ~360 seqs); raise for a tighter C
MAX_LEN      = 3072   # truncate long traces (bounds time/memory; the covariance converges fast)
SEED         = 0

# --- locate the snapshot tokens dir (auto, like the diagnostic finds the adapter) ---
cands = (glob.glob("/kaggle/input/**/04-08-16-14/tokens", recursive=True)
         + glob.glob("**/04-08-16-14/tokens", recursive=True)
         + glob.glob("/kaggle/working/**/04-08-16-14/tokens", recursive=True))
assert cands, "Could not find snapshot .../04-08-16-14/tokens -- set SNAP_TOKENS manually."
SNAP_TOKENS = cands[0]
SNAP_INDEX  = os.path.join(os.path.dirname(SNAP_TOKENS), "logprobs", "index.jsonl")
print("snapshot tokens:", SNAP_TOKENS)

# --- stratified sample of base problem ids (dedupe the byte-identical -p0/-dN copies) ---
_dup = re.compile(r"-(?:p|d)\d+$")
by_cat, seen = collections.defaultdict(list), set()
with open(SNAP_INDEX) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        pid = _dup.sub("", r["problem_id"])
        if pid in seen:
            continue
        seen.add(pid)
        by_cat[r["category"]].append(pid)

rng = random.Random(SEED)
sample = []
for cat, pids in sorted(by_cat.items()):
    rng.shuffle(pids)
    sample += [(cat, p) for p in pids[:PER_CATEGORY]]
rng.shuffle(sample)
print("per-category available:", {c: len(p) for c, p in sorted(by_cat.items())})
print(f"calibration sequences: {len(sample)}")

def load_tokens(pid):
    p = os.path.join(SNAP_TOKENS, pid, "synthetic.json")
    return json.load(open(p)).get("tokens") if os.path.exists(p) else None

# --- load base model ---
print("loading base model (this is the heavy step) ...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, low_cpu_mem_usage=True,
)
model.eval()
dev = next(model.parameters()).device

# --- hook every Mamba mixer.in_proj input; accumulate x^T x per module (fp32, on-device) ---
cov, cnt = {}, collections.Counter()
def mk_hook(name):
    def hook(mod, inp, out):
        x = inp[0].detach().reshape(-1, inp[0].shape[-1]).float()  # [tokens, hidden]
        g = x.T @ x
        cov[name] = g if name not in cov else cov[name] + g
        cnt[name] += x.shape[0]
    return hook

handles = [m.register_forward_hook(mk_hook(n))
           for n, m in model.named_modules() if n.endswith("mixer.in_proj")]
assert handles, "No mixer.in_proj modules found -- check model architecture / module naming."
print(f"hooked {len(handles)} in_proj modules")

# --- forward passes only (no grad, no generation) ---
done = 0
with torch.no_grad():
    for cat, pid in sample:
        toks = load_tokens(pid)
        if not toks:
            continue
        model(input_ids=torch.tensor([toks[:MAX_LEN]], device=dev))
        done += 1
        if done % 50 == 0:
            print(f"  {done}/{len(sample)} sequences")
for h in handles:
    h.remove()

# --- mean second moment + save (scale-free; converter applies a relative ridge) ---
C = {n: (cov[n] / max(cnt[n], 1)).cpu().float() for n in cov}
torch.save(C, COV_OUT)
print(f"\nsaved {len(C)} covariances -> {COV_OUT}")
print(f"tokens accumulated per layer: min {min(cnt.values())}  max {max(cnt.values())}")

del model
import gc; gc.collect(); torch.cuda.empty_cache()


In [ ]:
import os, re, torch
import tinker_cookbook.weights._adapter as A

FORCED_FUSED_RANK = 32
COV_PATH = "/kaggle/working/in_proj_cov.pt"  # produced by the calibration cell (optional)
RIDGE = 1e-3                                  # relative ridge on the input covariance

# If calibration ran, load per-layer input covariances => activation-aware (functional) SVD.
# Otherwise COV is empty and every layer falls back to plain weight-Frobenius SVD (= original code).
COV = torch.load(COV_PATH) if os.path.exists(COV_PATH) else {}
print(f"activation-aware truncation: {'ON' if COV else 'OFF (plain SVD)'}  ({len(COV)} covariances)")


def _lookup_cov(model_key_no_weight):
    """Find the covariance for a fused in_proj target. Exact match first, then by layer index."""
    if model_key_no_weight in COV:
        return COV[model_key_no_weight]
    m = re.search(r"layers\.(\d+)\..*in_proj$", model_key_no_weight)
    if m:
        idx = m.group(1)
        for k in COV:
            if re.search(rf"layers\.{idx}\..*in_proj$", k):
                return COV[k]
    return None


def _whitening_factor(C, ridge=RIDGE):
    """Right whitening factor W of C = E[x x^T] (C ~= W W^T), with relative ridge for safety."""
    C = C.double()
    C = 0.5 * (C + C.T)                                   # symmetrize
    eps = ridge * torch.diag(C).mean().clamp_min(1e-12)  # scale-free ridge
    C = C + eps * torch.eye(C.shape[0], dtype=C.dtype, device=C.device)
    try:
        W = torch.linalg.cholesky(C)                      # lower-tri, W @ W^T = C
    except Exception:                                    # fallback if not PD
        ev, V = torch.linalg.eigh(C)
        W = V @ torch.diag(torch.sqrt(ev.clamp_min(eps)))
    return W.float(), torch.linalg.inv(W).float()


def _compress_lora_pair_to_rank(B, A_mat, rank, C=None):
    """Best rank-`rank` factorization of delta = B @ A.
    C is None  -> Eckart-Young in unweighted Frobenius (identical to the original code).
    C given    -> minimizes E_x ||(delta - delta_hat) x||^2 via whitened SVD (un-whitened back)."""
    B, A_mat = B.float(), A_mat.float()
    delta = B @ A_mat                                     # [out_dim, in_dim]

    if C is None:
        U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
        U, S, Vh = U[:, :rank], S[:rank], Vh[:rank, :]
        sr = torch.sqrt(S.clamp_min(0))
        return (U * sr).to(B.dtype).contiguous(), (sr.unsqueeze(1) * Vh).to(A_mat.dtype).contiguous()

    C = C.to(delta.device)                               # keep all whitening math on delta's device
    W, W_inv = _whitening_factor(C)
    U, S, Vh = torch.linalg.svd(delta @ W, full_matrices=False)   # whitened: ||delta W - .||_F
    U, S, Vh = U[:, :rank], S[:rank], Vh[:rank, :]
    sr = torch.sqrt(S.clamp_min(0))
    B_new = U * sr.unsqueeze(0)                           # [out_dim, rank]
    A_new = (sr.unsqueeze(1) * Vh) @ W_inv                # un-whiten -> raw input basis [rank, in_dim]
    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous()


def patched_merge_fused_projections(
    fused_model_key,
    adapter_layer_prefix,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    assert component_order is not None

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts, comp_slices, merged_rank, row_offset = [], [], 0, 0
    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(
                f"Missing component {comp_name!r} for fused target {fused_model_key!r}"
            )
        lora_A, lora_B = comp_by_name[comp_name]
        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + lora_B.shape[0], lora_A.shape[0]))
        row_offset += lora_B.shape[0]
        merged_rank += lora_A.shape[0]

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim, merged_rank, dtype=merged_lora_A.dtype, device=merged_lora_A.device
    )
    rank_offset = 0
    for i, (row_start, row_end, r) in enumerate(comp_slices):
        _, lora_B = comp_by_name[component_order[i]]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    # Force fused modules back down to rank 32 (competition cap). Activation-aware if a
    # covariance is available for this layer, else plain Eckart-Young SVD.
    final_rank = merged_rank
    if merged_rank > FORCED_FUSED_RANK:
        C = _lookup_cov(fused_model_key.removesuffix(".weight"))
        mode = "act-aware" if C is not None else "plain"
        merged_lora_B, merged_lora_A = _compress_lora_pair_to_rank(
            merged_lora_B, merged_lora_A, FORCED_FUSED_RANK, C=C
        )
        final_rank = FORCED_FUSED_RANK
        print(f"  {fused_model_key}: {merged_rank}->{FORCED_FUSED_RANK} ({mode})")

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"
    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank


# monkey-patch
A._merge_fused_projections = patched_merge_fused_projections
print("patched:", A._merge_fused_projections.__name__)


In [ ]:
from tinker_cookbook import weights

weights.build_lora_adapter(
    base_model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
    adapter_path="/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    output_path="/kaggle/working/nemotron-adapter-ready-to-submit",
)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/submission', 'zip', '/kaggle/working/nemotron-adapter-ready-to-submit')